# 🔬 Module 5.2: CNN From Scratch

## Convolutional Neural Networks — From Math to Implementation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_05_Deep_Learning_For_Images/02_CNN_From_Scratch/cnn_from_scratch.ipynb)

---

## 🎯 Learning Objectives

1. Understand convolutional layers mathematically (connecting to Module 2 filters!)
2. Master pooling operations, stride, and padding with exact formulas
3. Implement a CNN forward pass from scratch in NumPy
4. Build the same CNN in PyTorch and compare
5. Visualize feature maps at each layer

---

In [ ]:
# Google Colab Setup
!pip install numpy matplotlib torch torchvision scikit-learn -q

In [ ]:
# Download REAL open-source datasets (TINY - under 250MB total)
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

# MNIST (handwritten digits - 11MB)
mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# CIFAR-10 (real photographs - 170MB)
transform_cifar = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))])
cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_cifar)
cifar_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_cifar)

# FashionMNIST (clothing items - 30MB, great for transfer learning!)
fashion_train = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
fashion_test = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

print(f"✅ MNIST: {len(mnist_train)} train + {len(mnist_test)} test (28x28 grayscale)")
print(f"✅ CIFAR-10: {len(cifar_train)} train + {len(cifar_test)} test (32x32 RGB)")
print(f"✅ FashionMNIST: {len(fashion_train)} train + {len(fashion_test)} test (28x28)")
print(f"   Classes: {cifar_train.classes}")
print(f"\n📦 Total download: ~211MB (NO STL-10 needed!)")

## Deep Derivation: Convolutional Layer Mathematics

### Step 1: Convolution Output Dimensions
Input: $H_{in} \times W_{in} \times C_{in}$, Kernel: $k \times k$, Stride: $s$, Padding: $p$

$$H_{out} = \left\lfloor \frac{H_{in} + 2p - k}{s} \right\rfloor + 1$$

**Derivation:** The kernel can be placed at positions $0, s, 2s, \ldots$ Starting from $-p$ to $H_{in}+p-k$:
$$\text{Number of positions} = \left\lfloor\frac{(H_{in}+2p-k)}{s}\right\rfloor + 1 \quad \blacksquare$$

### Step 2: Parameter Count
$$\text{Params} = C_{out} \times (C_{in} \times k^2 + 1)$$

Derivation: Each of $C_{out}$ filters has $C_{in}$ channels of size $k \times k$, plus one bias.

### Step 3: Receptive Field (Layer $l$)
$$r_l = r_{l-1} + (k_l - 1) \times \prod_{i=1}^{l-1} s_i$$

**Proof by induction:**
- Base: $r_0 = 1$ (single pixel)
- Step: Layer $l$ sees $k_l$ positions in layer $l-1$, each separated by stride product.

### Step 4: Translation Equivariance Proof
**Claim:** Convolution commutes with translation.

Let $T_\tau f(x) = f(x - \tau)$ be translation. Then:
$$(T_\tau f * g)(x) = \int f(t-\tau) g(x-t) dt$$
$$\overset{u=t-\tau}{=} \int f(u) g(x-\tau-u) du = (f*g)(x-\tau) = T_\tau(f*g)(x) \quad \blacksquare$$

**This means:** if object moves in input, feature map moves the same amount!

### Step 5: Backpropagation Through Conv Layer
Given $\frac{\partial L}{\partial Y}$, compute $\frac{\partial L}{\partial X}$ and $\frac{\partial L}{\partial W}$:

$$\frac{\partial L}{\partial X} = \frac{\partial L}{\partial Y} * W_{\text{flipped}} \quad \text{(full convolution)}$$

$$\frac{\partial L}{\partial W_{c_{out}, c_{in}}} = X_{c_{in}} \star \frac{\partial L}{\partial Y_{c_{out}}} \quad \text{(cross-correlation)}$$

Where $\star$ denotes valid cross-correlation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from sklearn.datasets import load_digits
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print('All libraries loaded!')

---

## Part 1: From Module 2 Filters to Learnable Convolutions

### 1.1 Recall: Convolution in Image Processing (Module 2)

In Module 2, we applied **fixed** filters (Sobel, Gaussian) to images via discrete convolution:

$$(I * K)(i, j) = \sum_{m} \sum_{n} I(i - m, j - n) \cdot K(m, n)$$

These filters detected edges, blurred images, or sharpened features. But the filters were **hand-designed**.

### 1.2 CNNs: Filters That Learn

A CNN **learns** the optimal filters from data via backpropagation. The convolution operation is identical, but the kernel weights $K$ become **trainable parameters**:

$$Z(i, j) = \sum_{c=1}^{C_{\text{in}}} \sum_{m=0}^{k_h-1} \sum_{n=0}^{k_w-1} X(c, i+m, j+n) \cdot W(c, m, n) + b$$

where:
- $X \in \mathbb{R}^{C_{\text{in}} \times H \times W}$ = input volume (channels × height × width)
- $W \in \mathbb{R}^{C_{\text{in}} \times k_h \times k_w}$ = one filter
- $b \in \mathbb{R}$ = bias for this filter
- $Z(i, j)$ = one element of the output feature map

With $C_{\text{out}}$ filters, the output is $\mathbb{R}^{C_{\text{out}} \times H' \times W'}$.

In [ ]:
# Module 2 callback: fixed filters vs learnable filters
sample_image = load_digits().images[0]  # 8x8

# Hand-designed filters from Module 2
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=float)
gaussian = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], dtype=float) / 16

# Random "learnable" filters (what a CNN starts with)
np.random.seed(42)
random_filter = np.random.randn(3, 3) * 0.5

def convolve2d(image, kernel):
    """Simple 2D convolution (valid mode)."""
    h, w = image.shape
    kh, kw = kernel.shape
    out_h, out_w = h - kh + 1, w - kw + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            output[i, j] = np.sum(image[i:i+kh, j:j+kw] * kernel)
    return output

filters = [
    ('Sobel-X (fixed)', sobel_x),
    ('Sobel-Y (fixed)', sobel_y),
    ('Gaussian (fixed)', gaussian),
    ('Random (learnable!)', random_filter),
]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))

axes[0, 0].imshow(sample_image, cmap='gray')
axes[0, 0].set_title('Input (8×8)', fontsize=11)
axes[1, 0].axis('off')
axes[1, 0].text(0.5, 0.5, 'Image from\nModule 2!', ha='center', va='center',
                fontsize=14, style='italic')

for i, (name, kernel) in enumerate(filters):
    result = convolve2d(sample_image, kernel)
    axes[0, i + 1].imshow(result, cmap='RdBu_r')
    axes[0, i + 1].set_title(name, fontsize=10)
    axes[1, i + 1].imshow(kernel, cmap='RdBu_r')
    axes[1, i + 1].set_title(f'Kernel ({kernel.shape[0]}×{kernel.shape[1]})', fontsize=10)
    for r in range(kernel.shape[0]):
        for c in range(kernel.shape[1]):
            axes[1, i + 1].text(c, r, f'{kernel[r,c]:.2f}', ha='center', va='center',
                                fontsize=8, fontweight='bold')

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

axes[0, 0].set_ylabel('Feature Map', fontsize=12)
axes[1, 0].set_ylabel('Filter/Kernel', fontsize=12)
plt.suptitle('Module 2 → Module 5: From Fixed Filters to Learnable Convolutions',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 2: Convolution Mathematics in Detail

### 2.1 Output Dimensions

For input size $H \times W$, kernel size $k \times k$, padding $p$, and stride $s$:

$$H_{\text{out}} = \left\lfloor \frac{H - k + 2p}{s} \right\rfloor + 1$$

$$W_{\text{out}} = \left\lfloor \frac{W - k + 2p}{s} \right\rfloor + 1$$

### 2.2 Types of Padding

| Type | Value of $p$ | Output Size (stride=1) | Use Case |
|------|-------------|----------------------|----------|
| **Valid** (no padding) | $p = 0$ | $H - k + 1$ | Reduce dimensions |
| **Same** | $p = \lfloor k/2 \rfloor$ | $H$ (preserved) | Keep spatial dims |
| **Full** | $p = k - 1$ | $H + k - 1$ | Increase dimensions |

### 2.3 Parameter Count

For $C_{\text{in}}$ input channels, $C_{\text{out}}$ output channels, kernel size $k \times k$:

$$\text{Parameters} = C_{\text{out}} \times (C_{\text{in}} \times k \times k + 1) = C_{\text{out}} \times C_{\text{in}} \times k^2 + C_{\text{out}}$$

Compare with a fully-connected layer: $H \times W \times C_{\text{in}} \times H' \times W' \times C_{\text{out}}$ — **much larger!**

This parameter sharing is the key advantage of convolutions.

In [ ]:
def compute_output_dim(H, k, p, s):
    return (H - k + 2 * p) // s + 1

configs = [
    (32, 3, 0, 1, 'Valid, stride=1'),
    (32, 3, 1, 1, 'Same, stride=1'),
    (32, 3, 0, 2, 'Valid, stride=2'),
    (32, 5, 2, 1, 'Same (5×5), stride=1'),
    (32, 5, 2, 2, 'Same (5×5), stride=2'),
    (28, 3, 0, 1, 'MNIST valid'),
    (224, 7, 3, 2, 'ResNet first conv'),
]

print(f'{"Config":30s} | {"Input":6s} | {"Kernel":6s} | {"Pad":4s} | {"Stride":6s} | {"Output":6s}')
print('-' * 80)
for H, k, p, s, name in configs:
    out = compute_output_dim(H, k, p, s)
    print(f'{name:30s} | {H:6d} | {k:6d} | {p:4d} | {s:6d} | {out:6d}')

# Visualize stride and padding
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (s, p, title) in zip(axes, [(1, 0, 'Stride=1, Pad=0 (Valid)'),
                                      (1, 1, 'Stride=1, Pad=1 (Same)'),
                                      (2, 0, 'Stride=2, Pad=0')]):
    H_in = 5
    k = 3
    H_out = compute_output_dim(H_in, k, p, s)
    H_padded = H_in + 2 * p

    grid = np.zeros((H_padded, H_padded))
    grid[p:p + H_in, p:p + H_in] = np.random.rand(H_in, H_in)

    ax.imshow(grid, cmap='Blues', vmin=-0.3, vmax=1.0)

    for i in range(0, H_padded - k + 1, s):
        for j in range(0, H_padded - k + 1, s):
            rect = plt.Rectangle((j - 0.5, i - 0.5), k, k,
                                  linewidth=1.5, edgecolor='red', facecolor='none',
                                  linestyle='--', alpha=0.5)
            ax.add_patch(rect)

    if p > 0:
        for i in range(H_padded):
            for j in range(H_padded):
                if i < p or i >= H_in + p or j < p or j >= H_in + p:
                    ax.text(j, i, '0', ha='center', va='center', fontsize=9,
                            color='gray')

    ax.set_title(f'{title}\nInput: {H_in}×{H_in} → Output: {H_out}×{H_out}', fontsize=11)
    ax.set_xticks(range(H_padded))
    ax.set_yticks(range(H_padded))

plt.suptitle('Stride and Padding Visualization (red boxes = kernel positions)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 3: Pooling Layers

### 3.1 Max Pooling

$$\text{MaxPool}(i, j) = \max_{0 \leq m < k, \; 0 \leq n < k} X(i \cdot s + m, \; j \cdot s + n)$$

### 3.2 Average Pooling

$$\text{AvgPool}(i, j) = \frac{1}{k^2} \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} X(i \cdot s + m, \; j \cdot s + n)$$

### 3.3 Properties

- **No learnable parameters** — purely structural
- Provides **translation invariance**: small shifts in input don't change the output
- Reduces spatial dimensions by factor $s$ (typically $k = s = 2$)

In [ ]:
def max_pool(X, pool_size=2, stride=2):
    h, w = X.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            patch = X[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            output[i, j] = np.max(patch)
    return output

def avg_pool(X, pool_size=2, stride=2):
    h, w = X.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            patch = X[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            output[i, j] = np.mean(patch)
    return output


np.random.seed(42)
X_demo = np.random.randint(0, 10, (6, 6)).astype(float)

max_result = max_pool(X_demo, 2, 2)
avg_result = avg_pool(X_demo, 2, 2)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, data, title in [(axes[0], X_demo, f'Input ({X_demo.shape[0]}×{X_demo.shape[1]})'),
                          (axes[1], max_result, f'Max Pool 2×2 ({max_result.shape[0]}×{max_result.shape[1]})'),
                          (axes[2], avg_result, f'Avg Pool 2×2 ({avg_result.shape[0]}×{avg_result.shape[1]})')]:
    ax.imshow(data, cmap='YlOrRd', vmin=0)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax.text(j, i, f'{data[i,j]:.1f}', ha='center', va='center',
                    fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=12)
    ax.set_xticks(range(data.shape[1]))
    ax.set_yticks(range(data.shape[0]))

# Draw pool regions on input
colors = ['red', 'blue', 'green', 'purple', 'orange', 'cyan', 'magenta', 'yellow', 'brown']
idx = 0
for i in range(0, X_demo.shape[0], 2):
    for j in range(0, X_demo.shape[1], 2):
        rect = plt.Rectangle((j - 0.5, i - 0.5), 2, 2,
                              linewidth=3, edgecolor=colors[idx % len(colors)],
                              facecolor='none')
        axes[0].add_patch(rect)
        idx += 1

plt.suptitle('Pooling Operations: Spatial Downsampling', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 4: CNN Forward Pass from Scratch (NumPy)

### 4.1 Full Convolution Layer Implementation

We implement a complete convolutional layer that handles:
- Multiple input channels
- Multiple output channels (filters)
- Padding and stride
- Bias terms

In [ ]:
class Conv2D:
    """2D Convolution layer implemented from scratch."""

    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

        # He initialization
        scale = np.sqrt(2.0 / (in_channels * kernel_size * kernel_size))
        self.W = np.random.randn(out_channels, in_channels, kernel_size, kernel_size) * scale
        self.b = np.zeros(out_channels)

    def forward(self, X):
        """
        X: shape (batch, in_channels, H, W)
        Returns: shape (batch, out_channels, H_out, W_out)
        """
        batch_size, C, H, W = X.shape
        k = self.kernel_size
        s = self.stride
        p = self.padding

        # Pad input
        if p > 0:
            X_pad = np.pad(X, ((0, 0), (0, 0), (p, p), (p, p)), mode='constant')
        else:
            X_pad = X

        H_out = (H + 2 * p - k) // s + 1
        W_out = (W + 2 * p - k) // s + 1

        output = np.zeros((batch_size, self.out_channels, H_out, W_out))

        for b_idx in range(batch_size):
            for f in range(self.out_channels):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * s
                        w_start = j * s
                        receptive_field = X_pad[b_idx, :, h_start:h_start+k, w_start:w_start+k]
                        output[b_idx, f, i, j] = np.sum(receptive_field * self.W[f]) + self.b[f]

        return output

    def __repr__(self):
        params = self.out_channels * (self.in_channels * self.kernel_size**2 + 1)
        return (f'Conv2D({self.in_channels}, {self.out_channels}, '
                f'kernel={self.kernel_size}, stride={self.stride}, '
                f'pad={self.padding}) — {params} params')


class MaxPool2D:
    """Max pooling layer."""

    def __init__(self, pool_size=2, stride=2):
        self.pool_size = pool_size
        self.stride = stride

    def forward(self, X):
        batch_size, C, H, W = X.shape
        k = self.pool_size
        s = self.stride
        H_out = (H - k) // s + 1
        W_out = (W - k) // s + 1

        output = np.zeros((batch_size, C, H_out, W_out))
        for b_idx in range(batch_size):
            for c in range(C):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * s
                        w_start = j * s
                        output[b_idx, c, i, j] = np.max(
                            X[b_idx, c, h_start:h_start+k, w_start:w_start+k]
                        )
        return output


class ReLULayer:
    def forward(self, X):
        return np.maximum(0, X)


class Flatten:
    def forward(self, X):
        return X.reshape(X.shape[0], -1)


print('Conv2D, MaxPool2D, ReLU, Flatten layers defined!')

In [ ]:
# Build a simple CNN from scratch
conv1 = Conv2D(in_channels=1, out_channels=4, kernel_size=3, stride=1, padding=1)
relu1 = ReLULayer()
pool1 = MaxPool2D(pool_size=2, stride=2)

conv2 = Conv2D(in_channels=4, out_channels=8, kernel_size=3, stride=1, padding=1)
relu2 = ReLULayer()
pool2 = MaxPool2D(pool_size=2, stride=2)

flatten = Flatten()

print('NumPy CNN Architecture:')
print(f'  {conv1}')
print(f'  ReLU + MaxPool 2×2')
print(f'  {conv2}')
print(f'  ReLU + MaxPool 2×2')
print(f'  Flatten')

# Forward pass with a sample 8×8 image
digits = load_digits()
X_sample = digits.images[0:1]  # shape (1, 8, 8)
X_sample = X_sample[:, np.newaxis, :, :]  # shape (1, 1, 8, 8)
print(f'\nInput shape: {X_sample.shape}')

# Layer-by-layer forward pass
z1 = conv1.forward(X_sample)
a1 = relu1.forward(z1)
p1 = pool1.forward(a1)
print(f'After Conv1 + ReLU + Pool: {p1.shape}')

z2 = conv2.forward(p1)
a2 = relu2.forward(z2)
p2 = pool2.forward(a2)
print(f'After Conv2 + ReLU + Pool: {p2.shape}')

flat = flatten.forward(p2)
print(f'After Flatten: {flat.shape}')
print(f'\nThis {flat.shape[1]}-dimensional vector is the feature representation!')

### 4.2 Visualizing the Forward Pass Through Each Layer

In [ ]:
def visualize_cnn_forward(X, conv1, relu1, pool1, conv2, relu2, pool2):
    """Visualize feature maps at every stage of the CNN."""
    stages = [
        ('Input', X[:, 0]),  # remove channel dim for display
    ]

    z1 = conv1.forward(X)
    stages.append(('After Conv1', z1[0]))

    a1 = relu1.forward(z1)
    stages.append(('After ReLU1', a1[0]))

    p1 = pool1.forward(a1)
    stages.append(('After Pool1', p1[0]))

    z2 = conv2.forward(p1)
    stages.append(('After Conv2', z2[0]))

    a2 = relu2.forward(z2)
    stages.append(('After ReLU2', a2[0]))

    p2 = pool2.forward(a2)
    stages.append(('After Pool2', p2[0]))

    fig = plt.figure(figsize=(22, 12))
    gs = gridspec.GridSpec(len(stages), 8, figure=fig, hspace=0.5)

    for row, (name, fmaps) in enumerate(stages):
        if fmaps.ndim == 2:
            fmaps = fmaps[np.newaxis]

        n_maps = fmaps.shape[0]
        col_start = (8 - n_maps) // 2

        for i in range(n_maps):
            ax = fig.add_subplot(gs[row, col_start + i])
            ax.imshow(fmaps[i], cmap='viridis')
            ax.set_title(f'Ch {i}' if n_maps > 1 else name, fontsize=9)
            ax.axis('off')

        fig.text(0.02, 1 - (row + 0.5) / len(stages), f'{name}\n{fmaps.shape}',
                 fontsize=10, va='center', ha='left', fontweight='bold',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.suptitle('CNN Forward Pass: Feature Maps at Every Layer',
                 fontsize=15, fontweight='bold')
    plt.show()

visualize_cnn_forward(X_sample, conv1, relu1, pool1, conv2, relu2, pool2)

---

## Part 5: CNN in PyTorch

### 5.1 The Same Architecture in PyTorch

PyTorch automates gradient computation and provides optimized GPU kernels:

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 8×8 → 8×8
            nn.ReLU(),
            nn.MaxPool2d(2),                               # 8×8 → 4×4
            nn.Conv2d(16, 32, kernel_size=3, padding=1),   # 4×4 → 4×4
            nn.ReLU(),
            nn.MaxPool2d(2),                               # 4×4 → 2×2
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 2 * 2, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = SimpleCNN().to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

# Verify forward pass
x_test = torch.randn(1, 1, 8, 8).to(device)
output = model(x_test)
print(f'\nInput shape:  {x_test.shape}')
print(f'Output shape: {output.shape}')

### 5.2 Training the PyTorch CNN on Digits

In [ ]:
# Prepare data
digits = load_digits()
X_all = digits.images[:, np.newaxis, :, :].astype(np.float32)  # (N, 1, 8, 8)
y_all = digits.target.astype(np.int64)

# Normalize
X_all = (X_all - X_all.mean()) / (X_all.std() + 1e-8)

# Train/test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train).to(device)
y_train_t = torch.tensor(y_train).to(device)
X_test_t = torch.tensor(X_test).to(device)
y_test_t = torch.tensor(y_test).to(device)

# Training
model = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

train_losses = []
test_accs = []

for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    output = model(X_train_t)
    loss = criterion(output, y_train_t)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_out = model(X_test_t)
            test_pred = test_out.argmax(dim=1)
            test_acc = (test_pred == y_test_t).float().mean().item()
            test_accs.append(test_acc)
            print(f'Epoch {epoch+1:3d} — Loss: {loss.item():.4f} — Test Acc: {test_acc:.2%}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(train_losses, 'b-', linewidth=1)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training Loss')

axes[1].plot(range(10, 101, 10), test_accs, 'g-o', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Test Accuracy')
axes[1].set_ylim(0, 1.05)

plt.suptitle('PyTorch CNN Training on 8×8 Digits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.3 Feature Map Visualization (PyTorch)

In [ ]:
def visualize_pytorch_feature_maps(model, x):
    """Extract and visualize feature maps from each layer of the trained CNN."""
    model.eval()
    feature_maps = []
    layer_names = []

    current = x
    for name, layer in model.features.named_children():
        current = layer(current)
        if isinstance(layer, (nn.Conv2d, nn.ReLU, nn.MaxPool2d)):
            feature_maps.append(current.detach().cpu().numpy()[0])
            layer_names.append(f'{layer.__class__.__name__} → {list(current.shape[1:])}')

    n_stages = len(feature_maps)
    fig, axes = plt.subplots(n_stages, 8, figsize=(20, 3 * n_stages))
    if n_stages == 1:
        axes = axes[np.newaxis]

    for row, (fmaps, name) in enumerate(zip(feature_maps, layer_names)):
        n_show = min(fmaps.shape[0], 8)
        for col in range(8):
            ax = axes[row, col]
            if col < n_show:
                ax.imshow(fmaps[col], cmap='viridis')
                ax.set_title(f'Filter {col}', fontsize=9)
            ax.axis('off')
        axes[row, 0].set_ylabel(name, fontsize=10, rotation=0, labelpad=100, va='center')

    plt.suptitle('Trained CNN Feature Maps (what the network sees at each layer)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Visualize for a few test images
for idx in [0, 5, 10]:
    x_vis = X_test_t[idx:idx+1]
    pred = model(x_vis).argmax(dim=1).item()
    true = y_test[idx]
    print(f'\nImage {idx}: True={true}, Predicted={pred}')

    fig_in, ax_in = plt.subplots(1, 1, figsize=(3, 3))
    ax_in.imshow(X_test[idx, 0], cmap='gray')
    ax_in.set_title(f'Input: digit {true}', fontsize=12)
    ax_in.axis('off')
    plt.show()

    visualize_pytorch_feature_maps(model, x_vis)

### 5.4 Learned Filters Visualization

In [ ]:
# Extract learned Conv1 filters
conv1_weights = model.features[0].weight.detach().cpu().numpy()  # (16, 1, 3, 3)

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for i, ax in enumerate(axes.flat):
    if i < conv1_weights.shape[0]:
        w = conv1_weights[i, 0]
        ax.imshow(w, cmap='RdBu_r', interpolation='nearest')
        ax.set_title(f'Filter {i}', fontsize=9)
        for r in range(3):
            for c in range(3):
                ax.text(c, r, f'{w[r,c]:.2f}', ha='center', va='center',
                        fontsize=7, fontweight='bold')
    ax.axis('off')

plt.suptitle('Learned Conv1 Filters (compare with Module 2 hand-designed filters!)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Notice: Some filters resemble edge detectors (like Sobel from Module 2)!')
print('The network has learned useful feature extractors from data alone.')

---

## Part 6: Receptive Field and Hierarchical Features

### 6.1 Receptive Field

The **receptive field** of a neuron at layer $l$ is the region of the input image that influences its value. For a stack of $l$ layers each with kernel $k$ and stride $s = 1$:

$$r_l = r_{l-1} + (k_l - 1) \cdot \prod_{i=1}^{l-1} s_i$$

For our architecture (two 3×3 conv + 2×2 pool layers):

| Layer | Kernel | Stride | Receptive Field |
|-------|--------|--------|----------------|
| Conv1 | 3×3 | 1 | 3×3 |
| Pool1 | 2×2 | 2 | 4×4 |
| Conv2 | 3×3 | 1 | 8×8 |
| Pool2 | 2×2 | 2 | 10×10 |

### 6.2 Hierarchical Feature Learning

- **Layer 1**: Detects edges, corners, color gradients (local patterns)
- **Layer 2+**: Combines lower-level features into textures, object parts
- **Deep layers**: Learns object-level and scene-level representations

This hierarchy is exactly what makes CNNs powerful for **RL state representation** — they automatically extract meaningful features from raw pixel images!

In [ ]:
def compute_receptive_field(layers):
    """Compute receptive field growth through a CNN."""
    r = 1  # initial receptive field
    j = 1  # cumulative stride (jump)
    results = [('Input', 1, 1)]

    for name, k, s in layers:
        r = r + (k - 1) * j
        j = j * s
        results.append((name, r, j))

    return results

layers = [
    ('Conv1 (3×3, s=1)', 3, 1),
    ('Pool1 (2×2, s=2)', 2, 2),
    ('Conv2 (3×3, s=1)', 3, 1),
    ('Pool2 (2×2, s=2)', 2, 2),
]

rf_results = compute_receptive_field(layers)

print(f'{"Layer":25s} | {"Receptive Field":15s} | {"Stride (jump)":15s}')
print('-' * 60)
for name, r, j in rf_results:
    print(f'{name:25s} | {r:15d} | {j:15d}')

# Visualize receptive field growth
fig, ax = plt.subplots(figsize=(10, 5))
names = [r[0] for r in rf_results]
rfs = [r[1] for r in rf_results]

bars = ax.bar(names, rfs, color=['lightblue'] + ['steelblue'] * len(layers), edgecolor='k')
for bar, rf in zip(bars, rfs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f'{rf}×{rf}', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('Receptive Field Size (pixels)', fontsize=12)
ax.set_title('Receptive Field Growth Through CNN Layers', fontsize=14)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

---

## 🔑 Key Takeaways

| Concept | Key Insight |
|---------|------------|
| **Convolution** | Same operation as Module 2, but filters are **learned** from data |
| **Output size** | $\lfloor(H - k + 2p)/s\rfloor + 1$ |
| **Parameter sharing** | Each filter is applied across the entire image → far fewer parameters than FC |
| **Pooling** | Reduces spatial dimensions, adds translation invariance |
| **Receptive field** | Deeper layers see larger regions → hierarchical features |
| **For RL** | CNN features become the state representation in Deep RL (DQN uses raw pixel CNNs!) |

### Connection to RL

The original **DQN (Mnih et al., 2015)** that played Atari games used exactly this architecture:
- Raw pixels → Conv layers → FC layers → Q-values
- The CNN automatically learns to extract game-relevant features from raw frames!

---

## ➡️ Next: Module 5.3 — Feature Extraction

Now that we can build CNNs, we'll explore **what they learn** at each layer using Grad-CAM and intermediate representations — crucial for understanding RL state features.